In [6]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from astropy.constants import G, M_sun
from astropy import units as u

In [12]:
AU = (1 * u.AU).to(u.m).value
year = (1 * u.yr).to(u.s).value
G_val = G.value
M_sun_val  = M_sun.value

In [14]:
class Body:
    def __init__(self, name, mass, position, velocity): #__init__ is called automatically when a new object of the class is created
        self.name = name
        self.mass = mass
        self.position = np.array(position, dtype=np.float64)
        self.velocity = np.array(velocity, dtype=np.float64)

In [18]:
# Define a dictionary
planet_data = {
    "Sun": {
        "mass": M_sun_val
    },
    "Mercury": {
        "mass": 3.3011e23,
        "a": 0.38709893, # in AU
        "e": 0.205630
    },
    "Venus": {
        "mass": 4.867e24,
        "a": 0.72333199,
        "e": 0.006773
    },
    "Earth": {
        "mass": 5.972e24,
        "a": 1.00000011,
        "e": 0.016710
    },
    "Mars": {
        "mass": 6.417e23,
        "a": 1.52366231,
        "e": 0.093412
    },
    "Jupiter": {
        "mass": 1.898e27,
        "a": 5.20336301,
        "e": 0.048393
    },
    "Saturn": {
        "mass": 5.683e26,
        "a": 9.53707032,
        "e": 0.054150
    }
}

In [20]:
def build_system(include_planets):
    bodies = []
    bodies.append(Body("Sun", planet_data["Sun"]["mass"], [0, 0], [0, 0]))
    
    for name in include_planets:
        pdata = planet_data[name]
        a = pdata["a"] * AU
        e = pdata["e"]
        r_peri = a * (1 - e)
        v_peri = np.sqrt(G_val * M_sun_val * (1 + e) / (a * (1 - e)))
        pos = [0, r_peri]
        vel = [v_peri, 0]
        bodies.append(Body(name, pdata["mass"], pos, vel))
    
    return bodies

In [22]:
def compute_accelerations(bodies):
    accels = [np.zeros(2) for _ in bodies]
    for i, bi in enumerate(bodies):
        for j, bj in enumerate(bodies):
            if i != j:
                r_vec = bj.position - bi.position
                dist = np.linalg.norm(r_vec)
                accels[i] += G_val * bj.mass * r_vec / dist**3
    return accels

In [24]:
def flatten_state(bodies):
    state = []
    for body in bodies:
        state.extend(body.position)
        state.extend(body.velocity)
    return np.array(state)

def unflatten_state(state, bodies):
    for i, body in enumerate(bodies):
        body.position = state[4*i:4*i+2]
        body.velocity = state[4*i+2:4*i+4]

In [26]:
def derivatives(t, state, bodies):
    unflatten_state(state, bodies)
    accels = compute_accelerations(bodies)
    derivs = []
    for i, body in enumerate(bodies):
        derivs.extend(body.velocity)
        derivs.extend(accels[i])
    return np.array(derivs)


In [28]:
def compute_perihelion_angles(x, y):
    r = np.sqrt(x**2 + y**2)
    r_shift = np.roll(r, 1)
    minima = (r < r_shift) & (r < np.roll(r, -1))
    x_min = x[minima]
    y_min = y[minima]
    angles = np.arctan2(y_min, x_min)
    angles = np.unwrap(angles) 
    return angles, minima

In [46]:
from scipy.stats import linregress

def compute_precession_rate(angles, times):
    t_years = times / (365.25 * 24 * 3600)  
    slope_rad_per_year = linregress(t_years, angles).slope
    arcsec_per_century = np.degrees(slope_rad_per_year) * 3600 * 100
    return arcsec_per_century

In [48]:
def run_simulation(include_planets, total_years=100, steps=5000):
    bodies = build_system(include_planets)
    y0 = flatten_state(bodies)
    t_span = (0, total_years * year)
    t_eval = np.linspace(*t_span, steps)
    sol = solve_ivp(derivatives, t_span, y0, t_eval=t_eval, args=(bodies,), rtol=1e-9)
    
    names = [b.name for b in bodies]
    mercury_index = names.index("Mercury")
    x = sol.y[4 * mercury_index, :] / AU
    y = sol.y[4 * mercury_index + 1, :] / AU
    angles, mask = compute_perihelion_angles(x, y)
    precession = compute_precession_rate(angles, sol.t[mask])
    
    return precession, x, y, sol.t

In [50]:
planet_sets = [
    ["Mercury"],
    ["Mercury","Jupiter"],
    ["Mercury","Jupiter", "Venus"],
    ["Mercury","Jupiter", "Venus", "Earth"],
    ["Mercury","Jupiter", "Venus", "Earth", "Saturn"],
    ["Mercury","Jupiter", "Venus", "Earth", "Saturn", "Mars"]
]

In [52]:
results = []
for planets in planet_sets:
    rate, _, _, _ = run_simulation(planets, total_years=100)
    results.append((planets, rate))

In [53]:
print("Precession contribution (arcsec/century):")
base = results[0][1]
for planets, rate in results:
    print(f"{' + '.join(['Sun'] + planets):<45}: {rate:.2f}  (+{rate - base:.2f})")

Precession contribution (arcsec/century):
Sun + Mercury                                : 4810.91  (+0.00)
Sun + Mercury + Jupiter                      : 239773.62  (+234962.71)
Sun + Mercury + Jupiter + Venus              : 235580.25  (+230769.34)
Sun + Mercury + Jupiter + Venus + Earth      : 233521.50  (+228710.59)
Sun + Mercury + Jupiter + Venus + Earth + Saturn: 220020.92  (+215210.01)
Sun + Mercury + Jupiter + Venus + Earth + Saturn + Mars: 218162.69  (+213351.78)
